In [1]:
import urllib.request
import feedparser

import fitz
import os
import numpy as np

import re

import ollama
import faiss

from rank_bm25 import BM25Okapi

In [2]:
BASE_URL = "http://export.arxiv.org/api/query?"

def search_arxiv(query, max_results=50):
    url_dict={"search_query":query, "start":0, "max_results":max_results}
    url = BASE_URL + urllib.parse.urlencode(url_dict)
    #urlencode and glue it back

    retrieved = urllib.request.urlopen(url)
    parsed_url = feedparser.parse(retrieved)

    results=[]
    for i in parsed_url.entries:
        pdf_url = ''
        for link in i.links:
            if link.get('title') == 'pdf':
                pdf_url = link['href']
                break
        results.append({
            'title': i.title,
            'summary': i.summary, 
            'pdf_url': pdf_url,
            'published': i.published
        })

    return results

papers=[]
topics = {
    'quantization': 'cat:cs.LG AND all:"quantization" AND all:"neural network"',
    'pruning': 'cat:cs.LG AND all:"pruning" AND all:"neural network"',
    'distillation': 'cat:cs.LG AND all:"knowledge distillation"',
}
for i in topics:
    papers += search_arxiv(topics[i], max_results=10)


In [3]:
len(papers)

30

In [4]:
PAPERS_DIR = "papers"
paper_ids=[]

for paper in papers:
    paper_ids.append(paper['pdf_url'].split('/')[-1])

def download_pdf(pdf_url, paper_id):
    os.makedirs(PAPERS_DIR, exist_ok=True)
    urllib.request.urlretrieve(pdf_url, os.path.join(PAPERS_DIR, paper_id))


def extract_text(pdf_path):
    content=""
    doc = fitz.open(pdf_path)
    for page in doc:
        content += page.get_text()
    
    return content

In [5]:
for i in range(len(papers)):
    download_pdf(papers[i]['pdf_url'], paper_ids[i])
    papers[i]['text'] = extract_text(os.path.join(PAPERS_DIR, paper_ids[i]))


In [6]:
def clean_text(text):
    result_text = re.sub(r'\s+', ' ', text)
    result_text = re.sub(r'-\s', '', result_text)

    return result_text.strip()

for i in range(len(papers)):
    papers[i]['text'] = clean_text(papers[i]['text'])


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_paper(text, c_size=500, overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = c_size, chunk_overlap = overlap)
    chunk_text = text_splitter.split_text(text)

    return chunk_text

for i in range(len(papers)):
    papers[i]['chunks'] = chunk_paper(papers[i]['text'], 1000, 200)

In [8]:
papers[0]['chunks']

['Published as a conference paper at ICLR 2020 ADDITIVE POWERS-OF-TWO QUANTIZATION: AN EFFICIENT NON-UNIFORM DISCRETIZATION FOR NEURAL NETWORKS Yuhang Li †∗, Xin Dong§∗, Wei Wang † †National University of Singapore, §Harvard University loafyuhang@gmail.com, xindong@g.harvard.edu, wangwei@comp.nus.edu.sg ABSTRACT We propose Additive Powers-of-Two (APoT) quantization, an efﬁcient nonuniform quantization scheme for the bell-shaped and long-tailed distribution of weights and activations in neural networks. By constraining all quantization levels as the sum of Powers-of-Two terms, APoT quantization enjoys high computational efﬁciency and a good match with the distribution of weights. A simple reparameterization of the clipping function is applied to generate a better-deﬁned gradient for learning the clipping threshold. Moreover, weight normalization is presented to reﬁne the distribution of weights to make the training more stable and consistent. Experimental results show that our proposed'

In [9]:
papers[0]['text']

'Published as a conference paper at ICLR 2020 ADDITIVE POWERS-OF-TWO QUANTIZATION: AN EFFICIENT NON-UNIFORM DISCRETIZATION FOR NEURAL NETWORKS Yuhang Li †∗, Xin Dong§∗, Wei Wang † †National University of Singapore, §Harvard University loafyuhang@gmail.com, xindong@g.harvard.edu, wangwei@comp.nus.edu.sg ABSTRACT We propose Additive Powers-of-Two (APoT) quantization, an efﬁcient nonuniform quantization scheme for the bell-shaped and long-tailed distribution of weights and activations in neural networks. By constraining all quantization levels as the sum of Powers-of-Two terms, APoT quantization enjoys high computational efﬁciency and a good match with the distribution of weights. A simple reparameterization of the clipping function is applied to generate a better-deﬁned gradient for learning the clipping threshold. Moreover, weight normalization is presented to reﬁne the distribution of weights to make the training more stable and consistent. Experimental results show that our proposed m

In [10]:
all_chunks = []

for paper in papers:
    for chunk in paper['chunks']:
        all_chunks.append({
            'text':chunk,
            'paper_title':paper['title'],
            'pdf_url':paper['pdf_url']
        })

print(all_chunks[0])
print(len(all_chunks))

chunk_text = [c['text'] for c in all_chunks]

{'text': 'Published as a conference paper at ICLR 2020 ADDITIVE POWERS-OF-TWO QUANTIZATION: AN EFFICIENT NON-UNIFORM DISCRETIZATION FOR NEURAL NETWORKS Yuhang Li †∗, Xin Dong§∗, Wei Wang † †National University of Singapore, §Harvard University loafyuhang@gmail.com, xindong@g.harvard.edu, wangwei@comp.nus.edu.sg ABSTRACT We propose Additive Powers-of-Two (APoT) quantization, an efﬁcient nonuniform quantization scheme for the bell-shaped and long-tailed distribution of weights and activations in neural networks. By constraining all quantization levels as the sum of Powers-of-Two terms, APoT quantization enjoys high computational efﬁciency and a good match with the distribution of weights. A simple reparameterization of the clipping function is applied to generate a better-deﬁned gradient for learning the clipping threshold. Moreover, weight normalization is presented to reﬁne the distribution of weights to make the training more stable and consistent. Experimental results show that our p

In [11]:
#EMBEDDING

def embed_chunks(chunks):
    response = ollama.embed(
        model = 'nomic-embed-text',
        input = chunks
    )

    return response['embeddings']

embeddings = embed_chunks(chunk_text)

test_embeddings = embed_chunks(chunk_text[:5])
print(len(test_embeddings))


5


In [12]:
print(len(embeddings))

2210


In [13]:
#first, converting list to numpy
embeddings_np = np.array(embeddings, dtype='float32')
print("Embedding shape: ", embeddings_np.shape)

#FAISS

d = embeddings_np.shape[1] #dimension of first chunk vector
index = faiss.IndexFlatL2(d)
print(index.is_trained)

index.add(embeddings_np)
print(index.ntotal)


Embedding shape:  (2210, 768)
True
2210


In [14]:
#BM25
def build_bm25_index(all_chunks):
    tokenised_chunks = []
    for chunk in all_chunks:
        tokens = chunk['text'].lower().split()
        tokenised_chunks.append(tokens)
    
    bm25 = BM25Okapi(tokenised_chunks)

    return bm25

In [15]:
tokenised_query = "quantization neural network".lower().split()

bm25 = build_bm25_index(all_chunks)
bm25_scores = bm25.get_scores(tokenised_query)

print(len(bm25_scores))
print(max(bm25_scores))

2210
5.754649804800733


In [16]:
query = "how does quantization neural network work"

query_embedding = ollama.embed(
    model = 'nomic-embed-text',
    input = [query]
)['embeddings']
query_vec = np.array(query_embedding, dtype='float32')
distances, faiss_ranked = index.search(query_vec, k=10)
faiss_ranked = faiss_ranked[0]

tokenised_query = query.lower().split()
scores = bm25.get_scores(tokenised_query)
bm25_ranked = np.argsort(scores)[::-1][:10]

In [17]:
def reciprocal_rank_fusion(faiss_ranked, bm25_ranked, k=60):
    rrf_scores={}
    for rank, chunk_idx in enumerate(faiss_ranked):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1/(k+rank+1)
    for rank, chunk_idx in enumerate(bm25_ranked):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1/(k+rank+1)
    
    return rrf_scores

ranks = reciprocal_rank_fusion(faiss_ranked, bm25_ranked)
print(ranks)


{np.int64(559): 0.01639344262295082, np.int64(248): 0.016129032258064516, np.int64(407): 0.015873015873015872, np.int64(297): 0.015625, np.int64(290): 0.015384615384615385, np.int64(295): 0.015151515151515152, np.int64(294): 0.014925373134328358, np.int64(185): 0.014705882352941176, np.int64(449): 0.014492753623188406, np.int64(408): 0.014285714285714285, np.int64(410): 0.01639344262295082, np.int64(744): 0.016129032258064516, np.int64(655): 0.015873015873015872, np.int64(4): 0.015625, np.int64(1781): 0.015384615384615385, np.int64(688): 0.015151515151515152, np.int64(570): 0.014925373134328358, np.int64(3): 0.014705882352941176, np.int64(569): 0.014492753623188406, np.int64(86): 0.014285714285714285}


In [23]:
sorted_ranks = sorted(ranks.items(), key=lambda x: x[1], reverse=True)

top_chunks=[]
for chunk_idx, score in sorted_ranks[:5]:
    chunk_idx = int(chunk_idx)
    top_chunks.append(all_chunks[chunk_idx])

for c in top_chunks:
    print(c['paper_title'], '-', c['text'])

Neural Network Quantization for Efficient Inference: A Survey - Neural Network Quantization for Efficient Inference: A Survey Olivia Weng Dept. of Computer Science and Engineering University of California, San Diego oweng@ucsd.edu ABSTRACT As neural networks have become more powerful, there has been a rising desire to deploy them in the real world; however, the power and accuracy of neural networks is largely due to their depth and complexity, making them difficult to deploy, especially in resource-constrained devices. Neural network quantization has recently arisen to meet this demand of reducing the size and complexity of neural networks by reducing the precision of a network. With smaller and simpler networks, it becomes possible to run neural networks within the constraints of their target hardware. This paper surveys the many neural network quantization techniques that have been developed in the last decade. Based on this survey and comparison of neural network quantization techni

In [24]:
def generate_answer(query, top_chunks):
    context = "\n\n".join([c['text'] for c in top_chunks])

    prompt = f"""Answer the question using only the context below.
    
    Context:
    {context}

    Question : {query}

    Answer:"""

    response = ollama.generate(model='llama3.1:8b', prompt = prompt)
    return response['response']

answer = generate_answer(query, top_chunks)
print(answer)

Quantization of a neural network involves replacing its parameters, which are typically represented as floating-point numbers, with more compact formats such as integers (e.g., 8 bits). This reduces the storage requirements and computational complexity of the network. The goal is to preserve the architecture and performance of the original network while using lower-precision representations.

In practice, quantization involves several steps:

1. **Choosing a quantization scheme**: There are different types of quantization schemes, including uniform and non-uniform quantization, symmetric and asymmetric quantization, and different calibration methods.
2. **Quantizing the parameters**: The neural network's parameters are replaced with their compact representations using the chosen quantization scheme.
3. **Preserving model accuracy**: Techniques such as Quantization-Aware-Training (QAT) or Post Training Quantization (PTQ) are used to ensure that the quantized network preserves its origin